In [ ]:
!pip install beam-ds==2.8.1rc2

In [18]:
from beam import setup

✨ | Setting up the Beam environment for interactive use
🚀 | Standard modules will be automatically imported so you can use them without explicit import
🛸 | Beam library is loaded from path: /Users/elads/projects/planning-flows/.venv/lib/python3.11/site-packages/beam
🔥 | 11:10:15 (0:00:00.374266) | INFO     🗎 Beam logger (2.8.1c2): logs are saved to /Users/elads/beam_data/logs/ipykernel_launcher-20250626-111015.log (∫__init__.py:__getattr__-#143)
⏲ | Done importing packages. It took:  0.59 seconds


# Server Side

In [1]:
# import sys
# sys.path.insert(0, '/Users/elads/projects/beamds/')
# from beam import setup

# from beam import beam_server
# from beam.bayesian import HPOService
# from beam.bayesian import BayesianHPOServiceConfig

# hparams = BayesianHPOServiceConfig()
# bs = HPOService(hparams=hparams)
# beam_server(bs, non_blocking=True)

# Client Side

In [19]:
from pydantic import BaseModel, conint, confloat, conlist
from typing import Literal

In [20]:
class Context(BaseModel):
    prompt: str

class Hyperparameters(BaseModel):
    temperature: confloat(ge=0, le=2)
    n_samples: Literal[1, 3, 7]
    depth: Literal[1, 3, 7]
    model: Literal["gpt-4.1-nano", "gpt-4.1-mini", "gpt-4.1", "vllm-llama-33-70b-instruct", "vllm-jamba16-large-040325"]

In [21]:
x_scheme = Hyperparameters.model_json_schema()
c_scheme = Context.model_json_schema()

In [22]:
from sentence_transformers import SentenceTransformer
import nltk
from nltk.corpus import gutenberg

nltk.download('gutenberg', quiet=True)
# Required downloads
nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True)  # new tokenizer

model_embedding = SentenceTransformer('all-MiniLM-L6-v2')
sentences = gutenberg.sents()
l_sentences = len(sentences)

In [23]:
def objective(h: Hyperparameters, c: str) -> float:
    # Base value influenced by temperature
    temperature_effect = -(h.temperature - 1)**2 + 1

    # Effect based on number of samples
    sample_effect = np.log(h.n_samples)

    # Effect based on depth, penalizing too shallow or too deep
    depth_effect = -(h.depth - 3)**2 + 4

    # Effect based on model complexity (arbitrary assigned scores)
    model_scores = {
        "gpt-4.1-nano": 0.5,
        "gpt-4.1-mini": 0.7,
        "gpt-4.1": 0.9,
        "vllm-llama-33-70b-instruct": 1.0,
        "vllm-jamba16-large-040325": 0.85,
    }
    model_effect = model_scores[h.model]

    # Context effect using embedding norm as example
    embedding = model_embedding.encode(c)
    context_effect = -np.linalg.norm(embedding) / 100

    # Aggregate the effects into a single scalar
    scalar_result = (
        2.0 * temperature_effect +
        1.5 * sample_effect +
        1.2 * depth_effect +
        3.0 * model_effect +
        context_effect
    )

    noise = float(np.random.randn()) * 4.
    return scalar_result + noise

def random_sample_hyperparameters() -> Hyperparameters:
    return Hyperparameters(
        temperature=random.uniform(0, 2),
        n_samples=random.choice([1, 3, 7]),
        depth=random.choice([1, 3, 7]),
        model=random.choice([
            "gpt-4.1-nano", 
            "gpt-4.1-mini", 
            "gpt-4.1", 
            "vllm-llama-33-70b-instruct", 
            "vllm-jamba16-large-040325"
        ])
    )

def random_sample_context() -> str:
    sample_sentence = random.randint(0, l_sentences-1)
    prompt = ' '.join(sentences[sample_sentence])
    return prompt

In [24]:
def sample_data(n=100):
    x = [random_sample_hyperparameters() for _ in range(n)]
    c = [random_sample_context() for _ in tqdm(range(n))]
    y = [objective(xi, ci) for xi, ci in tqdm(zip(x, c))]
    
    x = [dict(xi) for xi in x]
    c = [{'prompt': ci} for ci in c]
    return x, y, c

## Initial dataset

In [25]:
x, y, c = sample_data(120)

In [26]:
bsc = resource('http://localhost:35000')

In [27]:
bsc.register('gwc', x_scheme, c_scheme)

{'name': 'gwc',
 'x_scheme': {'properties': {'temperature': {'maximum': 2,
    'minimum': 0,
    'title': 'Temperature',
    'type': 'number'},
   'n_samples': {'enum': [1, 3, 7], 'title': 'N Samples', 'type': 'integer'},
   'depth': {'enum': [1, 3, 7], 'title': 'Depth', 'type': 'integer'},
   'model': {'enum': ['gpt-4.1-nano',
     'gpt-4.1-mini',
     'gpt-4.1',
     'vllm-llama-33-70b-instruct',
     'vllm-jamba16-large-040325'],
    'title': 'Model',
    'type': 'string'}},
  'required': ['temperature', 'n_samples', 'depth', 'model'],
  'title': 'Hyperparameters',
  'type': 'object'},
 'c_scheme': {'properties': {'prompt': {'items': {'type': 'number'},
    'maxItems': 32,
    'minItems': 32,
    'title': 'Prompt',
    'type': 'array'}},
  'required': ['prompt'],
  'title': 'Context',
  'type': 'object'},
 'message': "Problem 'gwc' registered successfully.",
 'embedding_keys': ['prompt']}

## add initial examples

In [28]:
with Timer():
    bsc.add('gwc', x, y, c)

🔥 | 11:10:50 (0:00:34.990233) | INFO     🗎 Starting timer:  (∫utils_all.py:__enter__-#992)
🔥 | 11:11:16 (0:01:00.838220) | INFO     🗎 Timer  paused. Elapsed time: 25.846     Sec (∫utils_all.py:__exit__-#999)


## Sample 10 new candidates

In [29]:
with Timer():
    r = bsc.sample('gwc', n_samples=10)

🔥 | 11:11:16 (0:01:00.868630) | INFO     🗎 Starting timer:  (∫utils_all.py:__enter__-#992)
🔥 | 11:12:05 (0:01:50.413246) | INFO     🗎 Timer  paused. Elapsed time: 49.543     Sec (∫utils_all.py:__exit__-#999)


In [30]:
r

{'name': 'gwc',
 'method': 'query',
 'message': 'Generated 10 samples with acquisition value: tensor([1.6584], dtype=torch.float64, grad_fn=<SubBackward0>)',
 'samples': [{'temperature': 1.763194965198636,
   'n_samples': 3,
   'depth': 3,
   'model': 'gpt-4.1'},
  {'temperature': 0.5607639588415623,
   'n_samples': 3,
   'depth': 3,
   'model': 'gpt-4.1'},
  {'temperature': 0.18309783563017845,
   'n_samples': 3,
   'depth': 3,
   'model': 'gpt-4.1'},
  {'temperature': 1.8574809543788433,
   'n_samples': 3,
   'depth': 3,
   'model': 'gpt-4.1'},
  {'temperature': 1.9671317711472511,
   'n_samples': 3,
   'depth': 3,
   'model': 'gpt-4.1'},
  {'temperature': 0.5168514624238014,
   'n_samples': 3,
   'depth': 3,
   'model': 'gpt-4.1'},
  {'temperature': 1.4884190615266562,
   'n_samples': 3,
   'depth': 3,
   'model': 'gpt-4.1'},
  {'temperature': 1.9378300588577986,
   'n_samples': 3,
   'depth': 3,
   'model': 'gpt-4.1'},
  {'temperature': 0.058812983334064484,
   'n_samples': 7,
   '

## add more examples

In [31]:
x, y, c = sample_data(20)

In [32]:
with Timer():
    r = bsc.add('gwc', x, y, c)

🔥 | 11:12:06 (0:01:51.217832) | INFO     🗎 Starting timer:  (∫utils_all.py:__enter__-#992)
🔥 | 11:12:08 (0:01:52.800553) | INFO     🗎 Timer  paused. Elapsed time: 1.582      Sec (∫utils_all.py:__exit__-#999)


In [33]:
r

{'name': 'gwc',
 'method': 'add',
 'message': 'Model updated with 20 fantasy points. New points: 20, Total points: 140, Fit every N points: 100.'}

# Using requests

In [34]:
import requests

In [35]:
response = requests.post('http://localhost:35000/alg/client/register', 
                            json={'args': ['gwc', x_scheme, c_scheme], 'kwargs': {}})
print(response.json())

{'name': 'gwc', 'x_scheme': {'properties': {'temperature': {'maximum': 2, 'minimum': 0, 'title': 'Temperature', 'type': 'number'}, 'n_samples': {'enum': [1, 3, 7], 'title': 'N Samples', 'type': 'integer'}, 'depth': {'enum': [1, 3, 7], 'title': 'Depth', 'type': 'integer'}, 'model': {'enum': ['gpt-4.1-nano', 'gpt-4.1-mini', 'gpt-4.1', 'vllm-llama-33-70b-instruct', 'vllm-jamba16-large-040325'], 'title': 'Model', 'type': 'string'}}, 'required': ['temperature', 'n_samples', 'depth', 'model'], 'title': 'Hyperparameters', 'type': 'object'}, 'c_scheme': {'properties': {'prompt': {'items': {'type': 'number'}, 'maxItems': 32, 'minItems': 32, 'title': 'Prompt', 'type': 'array'}}, 'required': ['prompt'], 'title': 'Context', 'type': 'object'}, 'message': "Problem 'gwc' registered successfully.", 'embedding_keys': ['prompt']}


In [42]:
# requires a fix

# response = requests.post('http://localhost:35000/alg/client/add', 
#                             json={'args': ['gwc', x, y, c], 'kwargs': {}})
# print(response.json())